In [1]:
# ============================================
# 1. IMPORTS AND SETUP
# ============================================

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path

DEVICE = "cpu"
RESULTS_DIR = Path("../results")

In [2]:
# ============================================
# 2. MODEL DEFINITION
#    Projection Head (same as training)
# ============================================

class ProjectionHead(nn.Module):
    def __init__(self, in_dim, hidden, out_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.ReLU(), nn.Dropout(0.2), nn.Linear(hidden, out_dim),
        )
    def forward(self, x):
        return F.normalize(self.net(x), dim=-1)

In [3]:
# ============================================
# 3. LOAD DATA AND MODEL CHECKPOINT
# ============================================

text = np.load(RESULTS_DIR / "text_embeddings_aligned.npy")
audio = np.load(RESULTS_DIR / "audio_embeddings_aligned.npy")
video = np.load(RESULTS_DIR / "video_embeddings_aligned.npy")

ckpt = torch.load(RESULTS_DIR / "alignment_model.pt", map_location=DEVICE, weights_only=False)
idx_test = ckpt["idx_test"]

In [4]:
# ============================================
# 4. STANDARDIZE TEST DATA
#    (Using training stats from checkpoint)
# ============================================

def standardize(mat, mu, sd):
    return ((mat - mu) / sd).astype(np.float32)

text_s = standardize(text, ckpt["t_mu"], ckpt["t_sd"])
audio_s = standardize(audio, ckpt["a_mu"], ckpt["a_sd"])
video_s = standardize(video, ckpt["v_mu"], ckpt["v_sd"])

In [5]:
# ============================================
# 5. LOAD TRAINED PROJECTION HEADS
# ============================================

text_head = ProjectionHead(text.shape[1], 256, 128)
text_head.load_state_dict(ckpt["text_head"])
text_head.eval()

audio_head = ProjectionHead(audio.shape[1], 512, 128)
audio_head.load_state_dict(ckpt["audio_head"])
audio_head.eval()

video_head = ProjectionHead(video.shape[1], 512, 128)
video_head.load_state_dict(ckpt["video_head"])
video_head.eval()

ProjectionHead(
  (net): Sequential(
    (0): Linear(in_features=2304, out_features=512, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=512, out_features=128, bias=True)
  )
)

In [6]:
# ============================================
# 6. GENERATE TEST EMBEDDINGS
#    Project test data to shared space
# ============================================

with torch.no_grad():
    zt = text_head(torch.tensor(text_s[idx_test])).numpy()
    za = audio_head(torch.tensor(audio_s[idx_test])).numpy()
    zv = video_head(torch.tensor(video_s[idx_test])).numpy()

In [7]:
# ============================================
# 7. RETRIEVAL EVALUATION FUNCTION
# ============================================

def retrieval_topk_acc(query, gallery, k=1):
    sims = query @ gallery.T
    n = sims.shape[0]
    topk = np.argsort(-sims, axis=1)[:, :k]
    correct = np.array([i in topk[i] for i in range(n)])
    return correct.mean()

In [8]:
# ============================================
# 8. RUN EVALUATION
#    Cross-modal retrieval performance
# ============================================

n_test = len(idx_test)
print(f"Test set size: {n_test}")
print(f"Chance top-1: {1/n_test:.4f}   Chance top-5: {5/n_test:.4f}\n")

for name, q, g in [("text->audio", zt, za), ("text->video", zt, zv), ("audio->video", za, zv)]:
    print(f"{name:14s}  top1={retrieval_topk_acc(q,g,1):.4f}  top5={retrieval_topk_acc(q,g,5):.4f}")

Test set size: 1028
Chance top-1: 0.0010   Chance top-5: 0.0049

text->audio     top1=0.0428  top5=0.0982
text->video     top1=0.0438  top5=0.1226
audio->video    top1=0.0496  top5=0.1158


In [9]:
# ============================================
# 9. SAVE TEST EMBEDDINGS
# ============================================

np.savez(RESULTS_DIR / "test_aligned_embeddings.npz", idx_test=idx_test, zt=zt, za=za, zv=zv)
print(f"Saved test aligned embeddings to {RESULTS_DIR / 'test_aligned_embeddings.npz'}")

Saved test aligned embeddings to ..\results\test_aligned_embeddings.npz
